In [17]:
import os
import json
import requests
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

True

In [18]:
gemini = OpenAI(
    base_url= "https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("GOOGLE_API_KEY")
)

ollama = OpenAI(
    base_url= "http://localhost:11434/v1/",
    api_key = "ollama"
)

In [19]:
def fetch_article(url):
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36" }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        return response.text[:12000]
    except requests.RequestException as e:
        return f"Error fetching article: {e}"

In [20]:
def extract_json_data(raw_text):
    system_prompt = """
    Anda adalah sistem ekstraksi data fundamental saham. Temukan metrik kunci dari teks.
    Keluarkan HANYA format JSON valid.
    
    Contoh Output:
    {
      "perusahaan": "Nama Perusahaan",
      "laba_bersih": "Angka laba/rugi",
      "pertumbuhan": "Persentase",
      "poin_krusial": ["Poin 1", "Poin 2"]
    }
    """
    try:
        respoinse = gemini.chat.completions.create(
            model="gemini-3.1-flash-lite",
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Ekstrak data dari artikel berikut:\n\n{raw_text}"}
            ]
        )
        return respoinse.choices[0].message.content
    except Exception as e:
        return f"Error extracting data: {e}"

In [ ]:
def analyze_and_stream(url):
    # Jalankan Scraper
    raw_html = fetch_article(url)
    if "Error" in raw_html:
        yield raw_html
        return
        
    yield "### ⏳ Membaca data metrik keuangan (Menggunakan Gemini)..."
    
    # Jalankan Ekstraksi JSON
    json_string = extract_json_data(raw_html)
    
    yield f"### ✅ Data Fundamental Terkumpul:\n```json\n{json_string}\n```\n\n### ⏳ Menyusun Analisis Eksekutif (Memanggil Llama 3.2)...\n"
    
    # Jalankan Analisis Sentimen & Fondasi Finansial Lokal
    system_prompt = """
    Anda adalah analis fundamental saham senior. Baca data JSON yang diberikan.
    Berikan analisis yang tajam, objektif, dan profesional mengenai prospek masa depan perusahaan.
    Gunakan format Markdown yang rapi (gunakan poin-poin).
    """
    
    stream = ollama.chat.completions.create(
        model="llama3.2",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Analisis data berikut: {json_string}"}
        ],
        stream=True
    )
    
    # Penanganan efek typewriter ke antarmuka Gradio
    final_output = f"### Data Fundamental Terkumpul:\n```json\n{json_string}\n```\n\n---\n\n"
    for chunk in stream:
        final_output += chunk.choices[0].delta.content or ""
        yield final_output

In [25]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# FinanceAI Radar")
    gr.Markdown("Masukkan tautan berita rilis laporan keuangan emiten untuk dianalisis oleh AI.")
    
    with gr.Row():
        url_input = gr.Textbox(label="URL Berita Keuangan", placeholder="Masukkan tautan berita di sini...")
        submit_btn = gr.Button("Analisis", variant="primary")
        
    output_markdown = gr.Markdown(label="Hasil Analisis")
    
    # Menghubungkan tombol klik ke fungsi generator streaming kita
    submit_btn.click(fn=analyze_and_stream, inputs=url_input, outputs=output_markdown)

# Jalankan aplikasi web langsung di dalam Notebook
demo.launch(server_name="0.0.0.0", inline=False)

/tmp/ipykernel_4563/3677142215.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://0.0.0.0:7864
* To create a public link, set `share=True` in `launch()`.
